# 1.0 Verify AWS and Bedrock Access

Run this notebook before building the graph in `1.1_build_graph.ipynb`. It checks the exact AWS paths that Module 1 needs: temporary Vocareum credentials, the `us-east-1` region, Claude Sonnet 4.6, and Amazon Nova multimodal embeddings.

You do not enter AWS keys here. Vocareum injects temporary credentials into the notebook environment. This notebook does not create resources, change permissions, subscribe models, or repair account access.


## Step 1: Confirm the AWS identity and region

The credential check stops instead of letting `boto3` fall back to another local profile. The identity and account number are safe to include in a support request. Secret values are never printed.


In [ ]:
import json
import os
from pathlib import Path

import boto3
from botocore.exceptions import BotoCoreError, ClientError
from dotenv import load_dotenv


def locate_notebooks_root():
    override = os.environ.get("WORKSHOP_NOTEBOOKS_DIR")
    if override:
        candidate = Path(override).expanduser().resolve()
        if (candidate / "workshop").is_dir():
            return candidate
        raise RuntimeError(
            "WORKSHOP_NOTEBOOKS_DIR must contain the workshop package"
        )

    start = Path.cwd().resolve()
    for candidate in (start, start / "notebooks", start.parent):
        if (candidate / "workshop").is_dir():
            return candidate
    raise RuntimeError(
        "Run from the repository root, notebooks/, or this module "
        "directory; or set WORKSHOP_NOTEBOOKS_DIR."
    )


NOTEBOOKS_ROOT = locate_notebooks_root()
REPO_ROOT = NOTEBOOKS_ROOT.parent
load_dotenv(NOTEBOOKS_ROOT / ".env")
load_dotenv(REPO_ROOT / ".env")
load_dotenv(REPO_ROOT / "CONFIG.txt")

BEARER_VARIABLE = "AWS_BEARER_TOKEN_BEDROCK"
bearer_token = os.environ.get(BEARER_VARIABLE)
if bearer_token is None:
    print(f"{BEARER_VARIABLE} is not set.")
    print("Bedrock calls will use this account's own credentials.")
elif not bearer_token.strip():
    raise RuntimeError(
        f"{BEARER_VARIABLE} is set to an empty value. The CONFIG.txt line was "
        "uncommented but no key was pasted in. Paste the Bedrock API key after "
        "the equals sign, or comment the line back out. An empty value is "
        "worse than no value at all, because botocore stops falling back to "
        "this account's normal credentials once the variable exists."
    )
else:
    print(f"{BEARER_VARIABLE} is set: {len(bearer_token)} characters, "
          f"ending {bearer_token[-4:]}.")
    print("Bedrock calls will use this key instead of this account's credentials.")

REQUIRED_REGION = "us-east-1"
region = (
    os.getenv("AWS_REGION")
    or os.getenv("AWS_DEFAULT_REGION")
    or boto3.Session().region_name
    or REQUIRED_REGION
)
session = boto3.Session(region_name=region)
credentials = session.get_credentials()

if credentials is None:
    raise RuntimeError("No AWS credentials are available. Start the Vocareum lab.")

frozen = credentials.get_frozen_credentials()
missing = [
    name
    for name, value in (
        ("access key", frozen.access_key),
        ("secret key", frozen.secret_key),
        ("session token", frozen.token),
    )
    if not value
]
if missing:
    raise RuntimeError("Missing temporary credential fields: " + ", ".join(missing))

identity = session.client("sts", region_name=region).get_caller_identity()
print(f"Account: {identity['Account']}")
print(f"Identity: {identity['Arn']}")
print(f"Region: {region}")

if region != REQUIRED_REGION:
    raise RuntimeError(
        f"This workshop requires {REQUIRED_REGION}; the session selected {region}."
    )

print("PASS: credentials, identity, and region are ready.")


## Step 2: Report the client and network conditions

This step prints facts instead of judging them. A failure in Step 3 is far easier to read when the botocore version, the proxy and certificate settings, and this kernel's outbound reach are already on the screen above it.

A Bedrock API key is sent as a bearer token, and botocore only does that from version 1.39.0 onward. An older botocore ignores the key and says nothing about it.

The outbound request goes to a host outside AWS. The fallback plan for a blocked account needs that path, so this notebook records whether it exists.


In [ ]:
import re
import socket
import ssl
import urllib.request
from urllib.parse import urlsplit

import botocore


BEARER_AUTH_FLOOR = (1, 39, 0)
EGRESS_URL = "https://pypi.org/simple/"
EGRESS_TIMEOUT_SECONDS = 10
NETWORK_VARIABLES = (
    "AWS_CA_BUNDLE",
    "HTTPS_PROXY",
    "HTTP_PROXY",
    "NO_PROXY",
    "REQUESTS_CA_BUNDLE",
    "SSL_CERT_FILE",
    "http_proxy",
    "https_proxy",
    "no_proxy",
)
SECRET_MARKERS = ("SECRET", "TOKEN", "PASSWORD", "KEY")


def version_tuple(text):
    """Read a version as numbers, ignoring any suffix such as the rc1 in 1.40.0rc1."""
    parts = []
    for piece in text.split("."):
        digits = re.match(r"\d+", piece)
        parts.append(int(digits.group()) if digits else 0)
    while len(parts) < len(BEARER_AUTH_FLOOR):
        parts.append(0)
    return tuple(parts)


def holds_a_secret(name):
    """True when a variable's name says its value must not be printed."""
    return any(marker in name.upper() for marker in SECRET_MARKERS)


def tls_issuer(url):
    """Name the authority that signed the certificate this kernel was served."""
    parts = urlsplit(url)
    host = parts.hostname
    port = parts.port or 443
    context = ssl.create_default_context()
    with socket.create_connection((host, port), EGRESS_TIMEOUT_SECONDS) as raw:
        with context.wrap_socket(raw, server_hostname=host) as secured:
            certificate = secured.getpeercert() or {}
    fields = dict(
        pair for group in certificate.get("issuer", ()) for pair in group
    )
    return fields.get("organizationName") or fields.get("commonName") or "unnamed"


floor_text = ".".join(str(part) for part in BEARER_AUTH_FLOOR)
print(f"botocore {botocore.__version__}")
print(f"boto3 {boto3.__version__}")
if version_tuple(botocore.__version__) >= BEARER_AUTH_FLOOR:
    print(f"PASS: botocore is at or above {floor_text}, so a bearer token is sent.")
else:
    print(f"FAIL: botocore is below {floor_text}.")
    print(f"  A Bedrock API key is ignored by this version. Install botocore "
          f"{floor_text} or later.")

print("\nProxy and certificate settings:")
scanned = set(NETWORK_VARIABLES)
scanned.update(
    name for name in os.environ if name.startswith("AWS_ENDPOINT_URL")
)
present = sorted(name for name in scanned if name in os.environ)
if present:
    for name in present:
        value = "set, value hidden" if holds_a_secret(name) else os.environ[name]
        print(f"  {name}={value}")
else:
    print("  none set")

print(f"\nOutbound request to {EGRESS_URL}:")
try:
    with urllib.request.urlopen(EGRESS_URL, timeout=EGRESS_TIMEOUT_SECONDS) as answer:
        print(f"  PASS: HTTP {answer.status}")
except Exception as error:
    # A diagnostic that raises hides the failure it was added to explain, so
    # every outcome here is printed rather than propagated.
    print(f"  FAIL: {type(error).__name__}: {error}")

try:
    print(f"  TLS issuer: {tls_issuer(EGRESS_URL)}")
except Exception as error:
    # Same reason as above. An unexpected issuer means the connection is being
    # intercepted, and no issuer at all means the handshake never completed.
    print(f"  TLS issuer unavailable: {type(error).__name__}: {error}")


## Step 3: Invoke the models Module 1 uses

This makes three small calls: Sonnet 4.6, streaming Sonnet 4.6, and one 1,024-dimension Nova embedding. A model appearing in the Bedrock catalog is not enough. Only a successful invocation proves the account can run the workshop.


In [ ]:
SONNET_MODEL_ID = "us.anthropic.claude-sonnet-4-6"
NOVA_MODEL_ID = "amazon.nova-2-multimodal-embeddings-v1:0"
EXPECTED_EMBEDDING_DIMENSIONS = 1024
MESSAGES = [
    {"role": "user", "content": [{"text": "Reply with one word: ready"}]}
]
INFERENCE_CONFIG = {"maxTokens": 32}
NOVA_REQUEST = {
    "taskType": "SINGLE_EMBEDDING",
    "singleEmbeddingParams": {
        "embeddingPurpose": "GENERIC_INDEX",
        "embeddingDimension": EXPECTED_EMBEDDING_DIMENSIONS,
        "text": {"truncationMode": "END", "value": "access check"},
    },
}


def failure_kind(error):
    code = error.response.get("Error", {}).get("Code", "Unknown")
    message = error.response.get("Error", {}).get("Message", "")
    lowered = message.lower()
    if "error 002" in lowered or "not allowed for this account" in lowered:
        return "ACCOUNT ENTITLEMENT", code, message
    if code in {"AccessDenied", "AccessDeniedException", "UnauthorizedOperation"}:
        return "IAM OR ORGANIZATION POLICY", code, message
    return "AWS SERVICE ERROR", code, message


def check(label, operation):
    try:
        detail = operation()
    except ClientError as error:
        kind, code, message = failure_kind(error)
        print(f"FAIL: {label}")
        print(f"  {kind}: {code}: {message}")
        return False
    except (BotoCoreError, KeyError, TypeError, ValueError) as error:
        print(f"FAIL: {label}")
        print(f"  CLIENT OR RESPONSE ERROR: {error}")
        return False
    print(f"PASS: {label} - {detail}")
    return True


runtime = session.client("bedrock-runtime", region_name=region)


In [ ]:
def invoke_sonnet():
    response = runtime.converse(
        modelId=SONNET_MODEL_ID,
        messages=MESSAGES,
        inferenceConfig=INFERENCE_CONFIG,
    )
    blocks = response["output"]["message"]["content"]
    text = "".join(block.get("text", "") for block in blocks).strip()
    if not text:
        raise ValueError("Sonnet returned no text")
    return f"answered {text[:40]!r}"


def stream_sonnet():
    response = runtime.converse_stream(
        modelId=SONNET_MODEL_ID,
        messages=MESSAGES,
        inferenceConfig=INFERENCE_CONFIG,
    )
    pieces = []
    for event in response["stream"]:
        delta = event.get("contentBlockDelta", {}).get("delta", {})
        if delta.get("text"):
            pieces.append(delta["text"])
    text = "".join(pieces).strip()
    if not text:
        raise ValueError("streaming Sonnet returned no text")
    return f"answered {text[:40]!r}"


def invoke_nova():
    response = runtime.invoke_model(
        modelId=NOVA_MODEL_ID,
        body=json.dumps(NOVA_REQUEST),
        contentType="application/json",
        accept="application/json",
    )
    payload = json.loads(response["body"].read())
    embeddings = payload.get("embeddings", [])
    vector = embeddings[0].get("embedding", []) if embeddings else []
    if len(vector) != EXPECTED_EMBEDDING_DIMENSIONS:
        raise ValueError(
            f"Nova returned {len(vector)} dimensions; "
            f"expected {EXPECTED_EMBEDDING_DIMENSIONS}"
        )
    return f"returned a {len(vector)}-dimension vector"


results = [
    check("Sonnet 4.6 InvokeModel", invoke_sonnet),
    check("Sonnet 4.6 streaming", stream_sonnet),
    check("Nova multimodal embeddings", invoke_nova),
]

if not all(results):
    raise RuntimeError(
        "Bedrock access is not ready. Copy the account, region, and failing line "
        "into the support request. Do not continue to 1.1."
    )

print("\nEnvironment is ready. Continue to 1.1_build_graph.ipynb.")


## Reading a failure

| Result | Meaning | Owner |
|---|---|---|
| `ACCOUNT ENTITLEMENT` and Error 002 | AWS blocks model use for this allocated account | Vocareum or AWS account-pool administrator |
| `IAM OR ORGANIZATION POLICY` | The session role or an organization policy denied the API action | Workshop template owner or Vocareum |
| Wrong region | The notebook is not using `us-east-1`, where the workshop models run | Lab configuration |
| Empty `AWS_BEARER_TOKEN_BEDROCK` | The CONFIG.txt line is uncommented with no key after it, so botocore sends an empty bearer header and never falls back | Whoever edited CONFIG.txt |
| botocore below `1.39.0` | This botocore cannot send a bearer token, so a Bedrock API key is ignored and the call runs on the blocked account | Reinstall the workshop requirements |
| All checks pass | The account can begin Module 1 | Continue to `1.1_build_graph.ipynb` |

Seeing a model in the Bedrock catalog does not prove it can be invoked. Error 002 is not repaired by changing this notebook, adding AWS keys, or adding another IAM allow statement to the workshop template.
